# RAG Evaluation with RAGAS
## A Hands-On Workshop

Building a RAG pipeline is only half the job. How do you know it is actually working well?
**RAGAS** (Retrieval Augmented Generation Assessment) gives you a suite of automatic metrics that score your pipeline without needing manually labelled data for every question.

By the end of this workbook you will:
- Build a simple RAG pipeline to evaluate
- Understand three core RAGAS metrics: **Faithfulness**, **Answer Relevance**, and **Context Recall**
- Run an evaluation and read the scores
- Know how to diagnose and improve a low-scoring pipeline

In [ ]:
# Install dependencies -- only runs on Google Colab.
# Local users: pip install ragas langchain-openai langchain-chroma python-dotenv
import sys

if "google.colab" in sys.modules:
    %pip install -q ragas langchain-openai langchain-chroma langchain-community python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    import getpass

    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

---

## Why Evaluate a RAG Pipeline?

RAG pipelines can fail in ways that are invisible from the outside:

| Failure mode | Symptom | Metric that catches it |
|---|---|---|
| LLM makes up facts not in the context | Confident but wrong answers | **Faithfulness** |
| Answer doesn't address the question | Technically true but irrelevant | **Answer Relevance** |
| Retriever misses necessary documents | Incomplete answers | **Context Recall** |
| Retriever pulls noisy unrelated chunks | Distracted LLM reasoning | **Context Precision** |

RAGAS scores each metric from 0 to 1. A score of 1 is perfect.
Running evaluation regularly -- not just once -- catches regressions when you change your chunking, model, or prompts.

---

## Part 1 -- Build the RAG Pipeline Under Test

We need a working RAG pipeline before we can evaluate it.
We will use a small hardcoded knowledge base about Python programming so the answers are easy to verify by hand.

In [ ]:
from langchain_core.documents import Document

# Knowledge base: 6 short documents about Python
DOCS = [
    Document(
        page_content="Python lists are ordered, mutable sequences. They support indexing, slicing, and methods like append(), extend(), and pop().",
        metadata={"source": "python-basics"},
    ),
    Document(
        page_content="Python dictionaries store key-value pairs. Keys must be hashable. Common methods include get(), keys(), values(), items(), and update().",
        metadata={"source": "python-basics"},
    ),
    Document(
        page_content="Python functions are defined with the def keyword. They support default arguments, *args for variable positional arguments, and **kwargs for keyword arguments.",
        metadata={"source": "python-functions"},
    ),
    Document(
        page_content="List comprehensions provide a concise way to create lists: [expr for item in iterable if condition]. They are generally faster than equivalent for loops.",
        metadata={"source": "python-advanced"},
    ),
    Document(
        page_content="Python classes are defined with the class keyword. The __init__ method initializes instances. Python supports single and multiple inheritance.",
        metadata={"source": "python-oop"},
    ),
    Document(
        page_content="Python exceptions are handled with try/except/finally blocks. Common exceptions include ValueError, TypeError, KeyError, and IndexError.",
        metadata={"source": "python-errors"},
    ),
]

print(f"Knowledge base: {len(DOCS)} documents")
for doc in DOCS:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:60]}...")

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Build an in-memory vector store from the documents
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(DOCS, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Quick test -- confirm retrieval works before evaluation
test_results = retriever.invoke("How do Python lists work?")
print(f"Retrieved {len(test_results)} chunks for test query")
for r in test_results:
    print(f"  {r.page_content[:80]}...")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano")

prompt = ChatPromptTemplate.from_template(
    """
Answer the question using ONLY the context below. If the context does not contain the answer, say "I don't know".

Context:
{context}

Question: {question}
Answer:""".strip()
)


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Smoke test
answer = rag_chain.invoke("What methods do Python lists support?")
print("Answer:", answer)

---

## Part 2 -- The Evaluation Dataset

RAGAS needs a dataset of questions with **ground truth answers** to measure context recall.
For faithfulness and answer relevance it only needs the question — RAGAS generates its own reference internally.

Each row has:
- `user_input` — the question
- `reference` — the ideal answer (ground truth, written by you)
- `retrieved_contexts` — what the retriever actually returned
- `response` — what the RAG chain actually answered

We build this by running the pipeline on a set of test questions and capturing every intermediate value.

In [ ]:
# Test questions + ground truth answers (written by hand)
# Ground truth should reflect what the knowledge base actually says
TEST_SET = [
    {
        "question": "What methods do Python lists support?",
        "ground_truth": "Python lists support methods like append(), extend(), and pop(), as well as indexing and slicing.",
    },
    {
        "question": "How are Python dictionaries structured?",
        "ground_truth": "Python dictionaries store key-value pairs where keys must be hashable. Common methods include get(), keys(), values(), items(), and update().",
    },
    {
        "question": "How do you define a Python function with variable arguments?",
        "ground_truth": "Python functions are defined with def and support *args for variable positional arguments and **kwargs for keyword arguments.",
    },
    {
        "question": "What is a list comprehension?",
        "ground_truth": "A list comprehension creates a list using the syntax [expr for item in iterable if condition] and is generally faster than an equivalent for loop.",
    },
    {
        "question": "How does Python handle exceptions?",
        "ground_truth": "Python handles exceptions with try/except/finally blocks. Common exceptions include ValueError, TypeError, KeyError, and IndexError.",
    },
]

print(f"Evaluation dataset: {len(TEST_SET)} questions")

In [ ]:
# Run the pipeline on every test question and capture the full trace
eval_rows = []

for item in TEST_SET:
    question = item["question"]
    contexts = retriever.invoke(question)
    response = rag_chain.invoke(question)

    eval_rows.append(
        {
            "user_input": question,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in contexts],
            "response": response,
        }
    )
    print(f"  Q: {question[:55]}...")
    print(f"  A: {response[:80]}...\n")

print(f"\nCollected {len(eval_rows)} rows for evaluation")

---

## Part 3 -- Running RAGAS

RAGAS takes our evaluation rows and scores them against three metrics:

### Faithfulness
Measures whether every claim in the answer can be inferred from the retrieved context.
An LLM that hallucinates facts not present in the context will score low here.

```
Faithfulness = claims supported by context / total claims in answer
```

### Answer Relevance
Measures whether the answer actually addresses the question asked.
A correct but off-topic answer scores low.

```
Answer Relevance = similarity(generated questions from answer, original question)
```

### Context Recall
Measures whether the retrieved context covers what is needed to give the ground truth answer.
A retriever that consistently misses the right documents will score low here.

```
Context Recall = ground truth sentences supported by context / total ground truth sentences
```

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import answer_relevancy, context_recall, faithfulness

# Wrap the LLM and embeddings for RAGAS
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Convert our eval rows to a HuggingFace Dataset (RAGAS input format)
dataset = Dataset.from_list(eval_rows)

print("Dataset ready:")
print(dataset)

In [ ]:
# Run evaluation -- this makes LLM calls for each metric on each row
# Expect ~15-25 API calls total for 5 questions x 3 metrics
results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(results)

In [ ]:
# Convert to a DataFrame for easier reading
df = results.to_pandas()

print("Per-question breakdown:")
print(
    df[["user_input", "faithfulness", "answer_relevancy", "context_recall"]].to_string(index=False)
)

print("\nMean scores:")
for col in ["faithfulness", "answer_relevancy", "context_recall"]:
    print(f"  {col:<22} {df[col].mean():.3f}")

---

## Part 4 -- Reading and Acting on the Scores

### What a good score looks like

| Score | Interpretation |
|-------|----------------|
| 0.9 + | Production-ready for this question type |
| 0.7 - 0.9 | Acceptable; monitor for edge cases |
| 0.5 - 0.7 | Needs improvement; investigate the low rows |
| < 0.5 | Serious problem; check retrieval and prompts |

### Diagnosing low scores

**Low Faithfulness** -- the LLM is adding information not in the retrieved context.
Fix: tighten the prompt ("Answer using ONLY the context"), reduce temperature, or use a stronger model.

**Low Answer Relevance** -- answers drift off-topic.
Fix: add a system prompt that enforces conciseness and direct answers.

**Low Context Recall** -- the retriever is missing relevant documents.
Fix: increase `k`, reduce chunk size, try a different embedding model, or switch to hybrid search.

In [ ]:
# Find the worst-performing questions per metric
for metric in ["faithfulness", "answer_relevancy", "context_recall"]:
    worst = df.nsmallest(1, metric).iloc[0]
    print(f"\nLowest {metric}: {worst[metric]:.3f}")
    print(f"  Q: {worst['user_input']}")
    print(f"  A: {worst['response'][:120]}")

---

## Part 5 -- Exercises

Try these before looking at the answer key at the end of the notebook.

In [ ]:
# Exercise 1 -- Degrade and measure
# Change the retriever to k=1 (only fetch 1 chunk instead of 2)
# Re-run the evaluation. Does context_recall drop? By how much?

# TODO: create retriever_k1 with search_kwargs={"k": 1}
# TODO: rebuild rag_chain_k1 using retriever_k1
# TODO: re-run the eval loop and call evaluate()
# TODO: compare the new context_recall mean to the original


In [ ]:
# Exercise 2 -- Break faithfulness intentionally
# Change the prompt to remove the "ONLY" constraint:
#   "Answer the question. Use the context if helpful."
# Re-run the evaluation. Does faithfulness drop?

# TODO: define prompt_loose without the ONLY constraint
# TODO: rebuild rag_chain_loose using prompt_loose
# TODO: re-run the eval loop and evaluate()
# TODO: compare faithfulness scores


In [ ]:
# Exercise 3 -- Add your own document and test question
# Add a new Document to DOCS about Python generators or decorators.
# Add a matching question + ground_truth to TEST_SET.
# Rebuild the vectorstore, re-run the pipeline, and evaluate the new question.

# TODO: define new_doc = Document(page_content="...", metadata={"source": "..."})
# TODO: rebuild vectorstore with DOCS + [new_doc]
# TODO: add a question + ground_truth entry to TEST_SET
# TODO: run the full evaluation and check the new row's scores


---
---

# Answer Key

Review these only after attempting the exercises yourself.

### Exercise 1 -- Degrade and measure (k=1 vs k=2)

Reducing k from 2 to 1 cuts the retrieved context in half. Context Recall should drop because some questions require evidence from two chunks to give a complete answer.

In [ ]:
# Exercise 1 answer
retriever_k1 = vectorstore.as_retriever(search_kwargs={"k": 1})

rag_chain_k1 = (
    {"context": retriever_k1 | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

eval_rows_k1 = []
for item in TEST_SET:
    q = item["question"]
    ctxs = retriever_k1.invoke(q)
    resp = rag_chain_k1.invoke(q)
    eval_rows_k1.append(
        {
            "user_input": q,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in ctxs],
            "response": resp,
        }
    )

results_k1 = evaluate(
    Dataset.from_list(eval_rows_k1),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

df_k1 = results_k1.to_pandas()
print("k=2  context_recall:", df["context_recall"].mean().round(3))
print("k=1  context_recall:", df_k1["context_recall"].mean().round(3))

> **What to look for:** context_recall with k=1 should be lower than k=2 because fewer chunks are available to cover the ground truth. The gap shows exactly how much retrieval coverage you are trading away.

### Exercise 2 -- Break faithfulness intentionally

Removing the "ONLY" constraint lets the LLM blend its own knowledge with the retrieved context. Faithfulness drops because the LLM makes claims that cannot be verified against the retrieved chunks.

In [ ]:
# Exercise 2 answer
prompt_loose = ChatPromptTemplate.from_template(
    """
Answer the question. Use the context if helpful.

Context:
{context}

Question: {question}
Answer:""".strip()
)

rag_chain_loose = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_loose
    | llm
    | StrOutputParser()
)

eval_rows_loose = []
for item in TEST_SET:
    q = item["question"]
    ctxs = retriever.invoke(q)
    resp = rag_chain_loose.invoke(q)
    eval_rows_loose.append(
        {
            "user_input": q,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in ctxs],
            "response": resp,
        }
    )

results_loose = evaluate(
    Dataset.from_list(eval_rows_loose),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

df_loose = results_loose.to_pandas()
print("strict prompt  faithfulness:", df["faithfulness"].mean().round(3))
print("loose prompt   faithfulness:", df_loose["faithfulness"].mean().round(3))

> **What to look for:** faithfulness often drops with the loose prompt because the LLM adds context from its training data. This is the core argument for explicit grounding instructions in production RAG prompts.

### Exercise 3 -- Add your own document and test question

In [ ]:
# Exercise 3 answer -- generators example
new_doc = Document(
    page_content="Python generators use the yield keyword to produce values lazily. They are memory-efficient for large sequences and can be iterated with for loops or next().",
    metadata={"source": "python-advanced"},
)

DOCS_EXT = DOCS + [new_doc]
vectorstore_ext = Chroma.from_documents(DOCS_EXT, embedding=embeddings)
retriever_ext = vectorstore_ext.as_retriever(search_kwargs={"k": 2})

new_question = {
    "question": "What is a Python generator and how does it work?",
    "ground_truth": "Python generators use the yield keyword to produce values lazily and are memory-efficient for large sequences.",
}

rag_chain_ext = (
    {"context": retriever_ext | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

q = new_question["question"]
ctxs = retriever_ext.invoke(q)
resp = rag_chain_ext.invoke(q)

new_row = [
    {
        "user_input": q,
        "reference": new_question["ground_truth"],
        "retrieved_contexts": [c.page_content for c in ctxs],
        "response": resp,
    }
]

result_new = evaluate(
    Dataset.from_list(new_row),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print(result_new)

> **What to look for:** if all three scores are high, the new document was successfully indexed and retrieved. If context_recall is low, the retriever is not surfacing the new chunk -- try increasing k or checking that the document was added to the vectorstore correctly.

---

## What's Next?

- **Example 12** -- [Basic RAG Notebook](../12-basic-rag-notebook/README.md): the pipeline we evaluated, built from scratch
- **Example 17** -- Corrective RAG: automatically fix low-confidence retrievals
- **RAGAS docs** -- https://docs.ragas.io: additional metrics (context precision, answer correctness), CI integration

### Key takeaways

| Metric | What it measures | Low score means |
|--------|-----------------|----------------|
| Faithfulness | Answer grounded in retrieved context | LLM is hallucinating |
| Answer Relevance | Answer addresses the question | Answer drifts off-topic |
| Context Recall | Retriever covers the ground truth | Retriever misses key docs |
| Context Precision | Retrieved context is focused | Too much noise in chunks |

Treat RAGAS scores as a regression test. Run them every time you change your chunking strategy, embedding model, or prompt.